# H&M GraphSAGE seed50 M1~M4 · 개인계정 · 로컬 매epoch / Drive10epoch
기존 과학적 조건·출력·실행 신원 유지. 로컬 최신·직전 전체상태 저장 후 10epoch 및 최종epoch에 Drive 업로드를 시도합니다. 명시적 API 요청 제한은 업로드를 보류하고 학습을 계속합니다. 마지막 원격 저장 이후의 로컬 구간은 런타임 삭제 시 유실될 수 있습니다. 완료 결과와 원격 최신·직전 저장본은 유지합니다. 최종 전송 대기가 남으면 런타임을 종료하지 말고 `screen.flush_pending_checkpoints()`로 재시도하세요. 이 명령은 학습 종료 후에만 실행합니다.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import subprocess, sys, json, hashlib, time, uuid, shutil
import torch
assert torch.cuda.is_available(), 'L4 GPU 런타임으로 설정하세요.'
assert 'L4' in torch.cuda.get_device_name(0), '기존 실험과 동일한 L4로 실행하세요.'
drive.mount('/content/drive')
auth.authenticate_user()  # 두 승인 모두 개인계정 선택
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
owner = drive_api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
assert owner == 'azsabae@gmail.com', f'개인 Drive가 아닙니다: {owner}'
SOURCE_COMMIT = '17aff7a9b3a6b12448fa4fbca5fedd4e6977b30b'
REPO = Path('/content/clv-hm-graphsage-core4-s50-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
sys.path.insert(0,str(REPO))
import clv_hm_graphsage_core4_seed50 as screen
import clv_personal_graphsage_checkpoint as storage
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이 런타임에 이전 코드가 남아 있습니다.'
print('개인 Drive 여유 GB:', storage.backend().check_space(drive_api,initial=True)/1e9)
DATA = Path('/content/drive/MyDrive/논문/data')
OUT = DATA / 'results_v3_hm_graphsage_core4_seed50_personal_v1'
OUT.mkdir(parents=True,exist_ok=True)
# Mount와 API가 같은 개인 Drive인지 작은 고유 파일로 검증. 다른 계정에는 학습하지 않음.
probe = OUT / ('account_check_' + uuid.uuid4().hex + '.txt')
probe.write_text('GraphSAGE seed50 personal core4',encoding='utf-8')
expected_md5 = hashlib.md5(probe.read_bytes()).hexdigest()
for attempt in range(12):
    rows = drive_api.files().list(q=f"'me' in owners and trashed=false and name='{probe.name}'",
        fields='files(id,md5Checksum)',pageSize=10).execute().get('files',[])
    if len(rows)==1 and rows[0].get('md5Checksum')==expected_md5:
        break
    time.sleep(5)
else:
    raise RuntimeError('마운트와 API의 개인 Drive 일치 확인 실패. 학습하지 않습니다.')
print('GPU:', torch.cuda.get_device_name(0), '| seed50 M1~M4 | 출력:', OUT)


## 1. 입력 준비 — 학습 없음
개인 Drive의 원본 두 파일을 바이트 그대로 Colab 로컬 디스크에 복사합니다.
기존 참조 결과 폴더가 있으면 그대로 읽고, 없으면 아래 이름의 ZIP을 references 폴더에서 사용합니다.
ZIP도 없으면 **작은 결과 ZIP 두 개만** 업로드하도록 요청합니다. 원본 데이터 재변환·재학습은 하지 않습니다.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''): h.update(block)
    return h.hexdigest()
EXPECTED = {
 'transactions_train.parquet':'0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
 'articles.csv':'25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b'}
LOCAL = Path('/content/hm_graphsage_seed50_inputs')
LOCAL.mkdir(exist_ok=True)
for key,name in [('tx_path','transactions_train.parquet'),('item_meta_path','articles.csv')]:
    source=DATA/'raw/hm'/name
    target=LOCAL/name
    if not target.is_file() or sha256(target)!=EXPECTED[name]:
        assert source.is_file(), f'기존 개인 Drive 원본 파일 없음: {source}'
        temporary=target.with_suffix(target.suffix+'.partial')
        shutil.copyfile(source,temporary)
        assert sha256(temporary)==EXPECTED[name], f'원본 바이트 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key]=str(target)
defaults=screen.configure()
ref_specs=[
 (Path(defaults.m2_reference),'clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
 (Path(defaults.m4_reference),'clv_hm_m4c_lastweek_seed49_results.zip')]
refs=[]
for directory,name in ref_specs:
    archive=DATA/'references'/name
    refs.append(directory if directory.is_dir() else archive)
missing=[p for p in refs if not p.exists()]
if missing:
    from google.colab import files
    print('다음 기존 결과 ZIP만 업로드하세요:', [p.name for p in missing])
    uploaded=files.upload()
    for p in missing:
        assert p.name in uploaded, f'누락: {p.name}'
        p.parent.mkdir(parents=True,exist_ok=True)
        p.write_bytes(uploaded[p.name])
cfg=screen.configure(out_dir=str(OUT),m2_reference=str(refs[0]),m4_reference=str(refs[1]))
assert cfg.seeds==(50,) and cfg.epochs==300
prepared=screen.prepare(cfg)
assert set(prepared['data']['splits'])=={'test'}
assert prepared['data'].get('loss_w') is None
checkpoint_folder=storage.install(screen,drive_api,prepared)
print('사전점검 완료: M1~M4 / seed50 / 300epoch')
print('결과 폴더:', prepared['run_dir'])
print('체크포인트 폴더:', 'https://drive.google.com/drive/folders/'+checkpoint_folder)


## 2. M1 → M2 → M3 → M4 학습
첫3epoch의 첫5배치 계측과 각 epoch의 학습·저장 시간을 기록합니다.
고정 입력 준비는 한 번만 수행합니다. 실제 시간 단축률은 GPU 실행 후 확인합니다.
오류 시 결과 패킷은 미완료(partial)로 보관하며 완료로 표시하지 않습니다.

In [ ]:
# Storage-only patch; scientific source and prepared identity unchanged.
import hashlib, urllib.request, types
url='https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/86401a42299b14c2721b9c216dc9951621227f89/clv_graphsage_local10.py'
with urllib.request.urlopen(url,timeout=60) as r: payload=r.read()
assert hashlib.sha256(payload).hexdigest()=='75c58f9de745d06ebc85dd42eea667e7d981eb8a746899ae9d64c1cfe9a58a34'
local10=types.ModuleType('clv_graphsage_local10')
exec(compile(payload,'<verified-local10>','exec'),local10.__dict__)
local10.install(screen,drive_api,prepared)
assert screen.SafeProgressStore._local10
assert screen.ORDER==('M1','M2','M3','M4') and cfg.seeds==(50,)
result=screen.run(prepared)
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))
print('Drive 전송 대기:',list(screen.pending_checkpoints))


## 3. 저장 결과와 시간 확인 — 추가 학습·평가 없음
M1 대비 동일 seed50 비교입니다. 전체142지표·진단·학습이력을 ZIP에 보존합니다.
학습 중에는 이 셀을 실행하지 마세요.

In [ ]:
import pandas as pd
from IPython.display import display
result=screen.report(prepared)
display(result['absolute'].set_index('model')[list(screen.CORE)].T.reindex(columns=screen.ORDER))
comparison=result['comparison']
if not comparison.empty:
    display(comparison.query("reference == 'M1'").pivot(index='metric',columns='model',
        values='relative_change_pct').reindex(index=screen.CORE))
paths=sorted((prepared['run_dir']/'progress/timing').glob('*.csv'))
if paths:
    timing=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True)
    display(timing.groupby('model_id')[['train_seconds','save_seconds']].mean())
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))
print('seed50 1시드 결과. M5_A~D는 이 노트북에서 실행하지 않습니다.')


## 4. 결과 다운로드
ZIP은 성능·설정·진단·학습이력만 포함합니다. 대용량 재개 체크포인트는 별도 개인 Drive 폴더에 유지합니다.
다운로드가 멈추면 출력된 Drive 결과 폴더에서 ZIP을 직접 받으세요.

In [ ]:
from google.colab import files
output=screen.export_zip(prepared,'/content/hm_graphsage_core4_seed50_results.zip')
backup=prepared['run_dir']/Path(output).name
shutil.copyfile(output,backup)
assert sha256(output)==sha256(backup)
print('Drive ZIP:',backup)
files.download(output)
